# Event streaming

Các LangChain agent được xây dựng trên LangGraph, vì vậy chúng hỗ trợ cùng một streaming stack với các projection tập trung vào agent cho các message, tool call, state và các bản cập nhật tùy chỉnh.

Đối với hầu hết các use case của ứng dụng và frontend, hãy sử dụng **Event Streaming** thông qua `stream_events(..., version="v3")`. Event Streaming trả về một đối tượng run với các projection được định kiểu, do đó mỗi projection có thể được tiêu thụ độc lập thay vì phải phân tích cú pháp các tuple của stream-mode.

In [2]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố."""
    return f"Trời luôn nắng đẹp ở {city}!"


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[get_weather],
)

input = {
    "messages": [{"role": "user", "content": "Thời tiết ở SF như thế nào?"}],
}

stream = agent.stream_events(input, version="v3")

for message in stream.messages:
    for delta in message.text:
        print(delta, end="", flush=True)

final_state = stream.output

Thời tiết ở San Francisco hiện tại rất đẹp, trời luôn nắng!

## Những gì bạn có thể stream

| Projection            | Công dụng                                                                        |
| --------------------- | ---------------------------------------------------------------------------------|
| `for event in stream` | Các event của raw protocol với đầy đủ envelope và quyền truy cập vào mọi channel.|
| `stream.messages`     | Các stream message của model, mỗi stream tương ứng với một lệnh gọi LLM.         |
| `message.text`        | Các text delta và văn bản cuối cùng cho một message.                             |
| `message.reasoning`   | Các reasoning delta cho các model có cung cấp nội dung suy luận.                 |
| `message.tool_calls`  | Các chunk tham số của tool-call và các tool call đã hoàn tất.                    |
| `message.output`      | Đối tượng message cuối cùng sau khi lệnh gọi model hoàn tất.                     |
| `stream.values`       | Các bản snapshot state của agent.                                                |
| `stream.output`       | State cuối cùng của agent.                                                       |
| `stream.subgraphs`    | Các run graph lồng nhau (sub-agent và các subgraph thông thường).                |
| `stream.extensions`   | Các transformer projection tùy chỉnh.                                            |
| `stream.tool_calls`   | Vòng đời thực thi tool, các input, output delta, output cuối cùng và error.      |

`stream.messages` yield các đối tượng `ChatModelStream`. Mỗi stream message sẽ cung cấp các thuộc tính `.text`, `.reasoning`, `.tool_calls`, và `.output`. Các sync projection có thể được duyệt qua để lấy các delta trực tiếp và có thể trích xuất để lấy các giá trị cuối cùng: hãy sử dụng `str(message.text)` để lấy văn bản cuối cùng và `message.tool_calls.get()` cho các tool call đã hoàn tất.

## Các message của agent

Sử dụng `stream.messages` khi bạn muốn lấy output của model từ mỗi lệnh gọi LLM.

In [3]:
stream = agent.stream_events(input, version="v3")

for message in stream.messages:
    print(f"[{message.node}] ", end="")
    for delta in message.text:
        print(delta, end="", flush=True)

    full_message = message.output
    usage = full_message.usage_metadata
    if usage:
        print(usage)

[model] {'input_tokens': 57, 'output_tokens': 16, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}}
[model] Trời luôn nắng đẹp ở SF!{'input_tokens': 94, 'output_tokens': 8, 'total_tokens': 102, 'input_token_details': {'cache_read': 0}}


`message.output` cung cấp cho bạn AI message đã hoàn tất, bao gồm các block nội dung đặc thù của nhà cung cấp (provider-specific). Trong TypeScript, hãy sử dụng `message.usage` khi bạn chỉ cần số lượng token hoặc các metadata sử dụng (usage metadata) khác; trong Python, hãy đọc mức sử dụng từ `message.output.usage_metadata`.

## Nội dung suy luận

Nội dung suy luận có cùng cấu trúc với nội dung văn bản, nhưng nó chỉ khả dụng khi model được chọn phát ra các reasoning block.

In [4]:
stream = agent.stream_events(input, version="v3")

for message in stream.messages:
    for delta in message.reasoning:
        print(f"[đang suy nghĩ] {delta}", end="", flush=True)

    for delta in message.text:
        print(delta, end="", flush=True)

Thời tiết ở San Francisco hiện tại trời luôn nắng đẹp!

Hãy xem [hướng dẫn về suy luận](https://docs.langchain.com/oss/python/langchain/models#reasoning) và trang tích hợp của nhà cung cấp để biết thông tin chi tiết về cách cấu hình model.

## Tool call

Có hai tool-call projection hữu ích:

* `message.tool_calls` sẽ stream các chunk tham số của tool-call trong lúc model đang tạo ra tool call.
* `stream.tool_calls` sẽ stream vòng đời thực thi của tool sau khi tool call bắt đầu.

In [5]:
stream = agent.stream_events(input, version="v3")

for message in stream.messages:
    for chunk in message.tool_calls:
        print(f"chunk của tool call: {chunk}")

    finalized = message.tool_calls.get()
    if finalized:
        print(f"các tool call đã hoàn tất: {finalized}")

for call in stream.tool_calls:
    print(f"{call.tool_name}({call.input})")
    for delta in call.output_deltas:
        print(delta, end="", flush=True)
    print(call.output, call.error)

các tool call đã hoàn tất: [{'type': 'tool_call', 'id': 'call_302745', 'name': 'get_weather', 'args': {'city': 'San Francisco'}}]


## Stream các sub-agent

Khi một lệnh gọi `create_agent` kích hoạt một `create_agent` khác có tên (thường thông qua một wrapping tool), các event của agent bên trong sẽ chảy vào một namespace lồng nhau. Tham số `name=` mà bạn truyền cho `create_agent` sẽ giúp định danh cho agent bên trong đó trên stream, nhờ vậy bạn có thể lọc và dán nhãn cho từng agent.

Các sub-agent có tên sẽ xuất hiện trên projection `stream.subagents` chuyên dụng. Mỗi handle sẽ cung cấp các `.messages`, `.values`, `.tool_calls`, và `.output` của riêng agent bên trong, cộng với `.name` (giá trị `name=` bạn đã truyền) và `.cause` (tool call đã điều phối sub-agent). Do chỉ những run của `create_agent` có tên mới xuất hiện ở đây, bạn không cần phải tự lọc bỏ các subgraph thông thường.

In [6]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố cụ thể."""
    return f"Trời luôn nắng đẹp ở {city}!"


weather_agent = create_agent(
    model=init_chat_model("google_genai:gemini-3.5-flash-lite"),
    tools=[get_weather],
    name="weather_agent",
)


def call_weather(query: str) -> str:
    """Truy vấn weather agent."""
    result = weather_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].text


supervisor = create_agent(
    model=init_chat_model("google_genai:gemini-3.5-flash-lite"),
    tools=[call_weather],
    name="supervisor",
)

stream = supervisor.stream_events(
    {"messages": [{"role": "user", "content": "Thời tiết ở Boston như thế nào?"}]},
    version="v3",
)

for subagent in stream.subagents:
    print(f"{subagent.name}: ", end="")
    for message in subagent.messages:
        for token in message.text:
            print(token, end="", flush=True)
    print()

weather_agent: Thời tiết ở Boston hiện tại luôn nắng đẹp!


Các subgraph `StateGraph` thông thường được gọi từ một tool cũng sẽ xuất hiện trên `stream.subgraphs` — hãy thiết lập `name=` trong `.compile(name=...)` để nhận được một nhãn trong `subagent.graph_name`.

`stream.subagents` là chế độ xem tập trung vào các sub-agent `create_agent` có tên, trong khi `stream.subgraphs` bao trùm mọi graph lồng nhau. Hãy sử dụng bất kỳ tùy chọn nào phù hợp với UI của bạn.

## State và output cuối cùng

Sử dụng `stream.values` cho các bản snapshot của state và `stream.output` cho state cuối cùng của agent.

In [7]:
stream = agent.stream_events(input, version="v3")

for snapshot in stream.values:
    print(snapshot)

final_state = stream.output

{'messages': [HumanMessage(content='Thời tiết ở SF như thế nào?', additional_kwargs={}, response_metadata={}, id='ac7e058b-cedf-4c35-b678-514b6b5d180e')]}
{'messages': [HumanMessage(content='Thời tiết ở SF như thế nào?', additional_kwargs={}, response_metadata={}, id='ac7e058b-cedf-4c35-b678-514b6b5d180e'), AIMessage(content=[{'type': 'tool_call', 'id': 'call_327462', 'name': 'get_weather', 'args': {'city': 'San Francisco'}}], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "San Francisco"}'}, '__gemini_function_call_thought_signatures__': {'call_327462': 'El4KXAERTTIPAE7UTczIHkNiITmLu8se2nCWZM1qagILnWwSt49SDPwFbgHpgidIctGaeXtGrZmoGxSdEpAY5UiN4tJHuG4OtVBQRvJPkU9jugCwDkbll2JgR+YlOmhH'}}, response_metadata={'model_provider': 'google_genai', 'safety_ratings': [], 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'output_version': 'v1'}, id='lc_run--01a06526-7d1a-70a0-b25a-a0ee094bb772', tool_calls=[{'name': 'get_weather', 'args': {'city': '

## Nhiều projection

Để tiêu thụ đồng thời trong code async, hãy sử dụng `astream_events` cùng với `asyncio.gather`:

In [9]:
import asyncio

stream = await agent.astream_events(input, version="v3")

async def consume_messages():
    async for message in stream.messages:
        print(await message.text)

async def consume_tool_calls():
    async for call in stream.tool_calls:
        print(call.tool_name, call.input)

await asyncio.gather(consume_messages(), consume_tool_calls())


get_weather {'city': 'SF'}
Trời luôn nắng đẹp ở SF!


[None, None]

Đối với code đồng bộ (synchronous), hãy sử dụng `stream.interleave(...)` để thay thế:

In [10]:
stream = agent.stream_events(input, version="v3")

for name, item in stream.interleave("messages", "tool_calls", "values"):
    if name == "messages":
        print(item.text)
    elif name == "tool_calls":
        print(item.tool_name, item.input)
    elif name == "values":
        print(item)

{'messages': [HumanMessage(content='Thời tiết ở SF như thế nào?', additional_kwargs={}, response_metadata={}, id='5daa8c72-914e-4cc7-a22e-175ce09cb05f')]}

{'messages': [HumanMessage(content='Thời tiết ở SF như thế nào?', additional_kwargs={}, response_metadata={}, id='5daa8c72-914e-4cc7-a22e-175ce09cb05f'), AIMessage(content=[{'type': 'tool_call', 'id': 'call_228023', 'name': 'get_weather', 'args': {'city': 'SF'}}], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "SF"}'}, '__gemini_function_call_thought_signatures__': {'call_228023': 'El4KXAERTTIP4ECVBEY9OlppQQy70VS7AIqRvmiS+cNEkVnMHFP9OyozEVDVl8wpw40eeNm27EnTSjUJ6bi4RpFQ6H79fpWX+G5YxECINiu3L2x5xBn5Z6oX8U0Ny6sm'}}, response_metadata={'model_provider': 'google_genai', 'safety_ratings': [], 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'output_version': 'v1'}, id='lc_run--01a06529-21ab-7042-b440-4eccc953fc6c', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'call_228

Để truy cập các channel không được bộc lộ dưới dạng typed projection, hoặc để kiểm tra toàn bộ event envelope, hãy duyệt vòng lặp qua các event của raw protocol:

In [12]:
stream = agent.stream_events(input, version="v3")

for event in stream:
    print(event["method"], event["params"]["namespace"], event["params"]["data"])

values [] {'messages': [HumanMessage(content='Thời tiết ở SF như thế nào?', additional_kwargs={}, response_metadata={}, id='f7ff1e3d-23cd-483b-a238-6ad050ea5130')]}
messages [] ({'event': 'message-start', 'role': 'ai', 'id': 'lc_run--01a0652a-949d-75c1-9d93-c72417f6851f', 'metadata': {'provider': 'google_genai'}}, {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:a7d3ae9e-b828-1f23-23d6-dbd24ac1b11b', 'checkpoint_ns': 'model:a7d3ae9e-b828-1f23-23d6-dbd24ac1b11b', 'ls_provider': 'google_genai', 'ls_model_name': 'gemini-3.5-flash-lite', 'ls_model_type': 'chat', 'ls_temperature': None, 'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16', 'langchain-google-genai': '4.3.5'}, 'run_id': '01a0652a-949d-75c1-9d93-c72417f6851f'})
messages [] ({'event': 'content-block-start', 'index': 0, 'content': {'type': 'tool_call', '

## Các bản cập nhật tùy chỉnh

Hãy sử dụng các stream transformer tùy chỉnh khi ứng dụng của bạn cần một projection không được tích hợp sẵn, chẳng hạn như tiến trình truy xuất, artifact, hoặc các event đặc thù của domain.

In [ ]:
stream = agent.stream_events(
    input,
    version="v3",
    transformers=[ToolActivityTransformer],
)

for activity in stream.extensions["tool_activity"]:
    print(activity)

### Đăng ký các transformer trên middleware

<div class="alert alert-info">

Các transformer được đăng ký qua middleware yêu cầu phiên bản `langchain>=1.3.2`.

</div>

Middleware có thể khai báo các factory của stream transformer cùng với các hook và tool của nó. Cấu trúc của factory sẽ khác nhau tùy theo ngôn ngữ:

Hãy thiết lập thuộc tính `transformers` trên một subclass của `AgentMiddleware` thành một chuỗi các factory. Mỗi factory có cấu trúc `Callable[[tuple[str, ...]], StreamTransformer]` và được gọi dưới dạng `factory(scope)`, trong đó `scope` là một tuple phạm vi mini-mux (`()` đối với mux gốc, và không rỗng đối với các subgraph). Việc trả về một transformer mới cho mỗi lệnh gọi sẽ giúp giữ cho mỗi subgraph được cô lập.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware


class ToolActivityMiddleware(AgentMiddleware):
    transformers = (ToolActivityTransformer,)


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[get_weather],
    middleware=[ToolActivityMiddleware()],
)

Tại thời điểm compile, `create_agent` sẽ hợp nhất các factory được đăng ký qua middleware với bất kỳ thứ gì được truyền vào tham số `transformers=` của riêng nó. Thứ tự cuối cùng trên graph đã biên dịch là:

1. `ToolCallTransformer` được tích hợp sẵn.
2. Các factory được đăng ký qua middleware, theo thứ tự của middleware.
3. Các tham số `transformers=` do người gọi cung cấp từ `create_agent`.

Điều này giữ cho tool-call projection tích hợp sẵn nằm phía trước các transformer tiêu thụ, đồng thời trao quyền quyết định cuối cùng cho các entry do người gọi cung cấp.

`PIIMiddleware` tích hợp sẵn sử dụng hook này để làm mờ thông tin PII từ output trên đường truyền được stream. Với `apply_to_output=True`, transformer được đăng ký của nó sẽ xóa bỏ các PII được phát hiện từ text delta, tham số tool-call, output của tool và các snapshot của state trước khi chúng rời khỏi run. Điều này giúp chặn đứng kẽ hở mà chức năng redact cấp độ state `after_model` có thể vô tình để lọt thông tin PII thô tới người đọc trực tiếp của `stream_events(version="v3")`.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_output=True),
    ],
)

Hãy xem [PII detection)](https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection) để biết toàn bộ thông tin về cấu hình.

Hãy xem [Xây dựng projection của riêng bạn](https://docs.langchain.com/oss/python/langgraph/event-streaming#build-your-own-projection) để biết thông tin về transformer contract.

## Bài viết liên quan

* [Streaming](https://docs.langchain.com/oss/python/langchain/streaming) trình bày về các stream mode Pregel cấp thấp.
* [Xây dựng projection của riêng bạn](https://docs.langchain.com/oss/python/langgraph/event-streaming#build-your-own-projection) bao gồm việc viết các projection đặc thù cho ứng dụng.
* [Các pattern stream cho frontend](https://docs.langchain.com/oss/python/langchain/frontend/overview) trình bày các use case của UI được xây dựng trên streamed state.